# HippoRAG 2 on Colab — DeepSeek for OpenIE, NV-Embed-v2 locally

Builds a HippoRAG 2 index over MuSiQue and runs retrieval, scoring against the
same gold this repo uses everywhere else.

**Split of work.** OpenIE (~23k extraction calls) goes to the DeepSeek API, which is
cheap and needs no GPU. The encoder (NV-Embed-v2, 7B) runs locally, because nobody
serves it as an API and it is the part with no per-call cost.

**Before you start**
- Runtime → Change runtime type → **L4 GPU**. A T4 will not work (see the next cell).
- A DeepSeek API key from platform.deepseek.com.
- ~10 GB free on Google Drive for the index.

**On disconnects.** The index lives on Drive, and OpenIE results are cached, so
re-running the indexing cell after a dropout resumes rather than restarting. That is
the whole reason for the Drive mount.

In [ ]:
# --- GPU check. Stop here rather than OOM twenty minutes in. ---
import subprocess, sys
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(out or 'NO GPU — Runtime > Change runtime type > L4 GPU')
assert out, 'No GPU attached.'
name, mem = [x.strip() for x in out.split(',')]
mib = int(mem.split()[0])
# NV-Embed-v2 is 7B: ~15 GB of fp16 weights before activations. A 16 GB T4 cannot
# hold it, and Turing has no bf16 either.
assert mib >= 20000, (
    f'{name} has {mib} MiB. NV-Embed-v2 needs ~15 GB of weights plus activations; \n'
    'use an L4 (24 GB) or A100. A T4 will OOM on model load.')
print(f'ok: {name}, {mib} MiB')

In [ ]:
# --- Drive: the index and the OpenIE cache live here so a disconnect resumes ---
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/hipporag'   # parquet stores, graph, llm cache
os.makedirs(SAVE_DIR, exist_ok=True)
# HF cache stays LOCAL on purpose: reading 15 GB of weights through Drive's FUSE
# layer is slower than re-downloading them, and they are reproducible anyway.
os.environ['HF_HOME'] = '/root/.cache/huggingface'
print('index ->', SAVE_DIR)

In [ ]:
# --- Repo + submodule + patch ---
REPO   = 'https://github.com/pakhomovee/ysda-graph-rag.git'
BRANCH = 'qafd-edge-probe'
GITHUB_TOKEN = ''   # only if the repo is private

url = REPO.replace('https://', f'https://{GITHUB_TOKEN}@') if GITHUB_TOKEN else REPO
%cd /content
![ -d ysda-graph-rag ] || git clone -b $BRANCH $url ysda-graph-rag
%cd /content/ysda-graph-rag
!git pull --quiet && bash scripts/setup_hipporag.sh 2>&1 | tail -20

In [ ]:
# --- Dependencies: what THIS path needs, not their whole requirements.txt ---
# Deliberately NOT installed:
#   vllm==0.6.6.post1  ~2 GB, and we call an API rather than serving locally
#   gritlm==1.0.2      pulls sentence-transformers, whose 5.x imports torchcodec at
#                      module scope; torchcodec cannot load against Colab's torch
#                      2.5.1. It is a dependency of the GritLM backend, which we
#                      never select -- and the patch now imports backends lazily, so
#                      not installing it costs nothing.
#   torch==2.5.1       Colab already has it; reinstalling burns minutes per session.
!pip install -q 'transformers==4.45.2' 'openai>=1.91.0' tenacity tiktoken \
    python_igraph networkx pydantic pandas pyarrow einops nest_asyncio \
    tqdm scipy requests 2>&1 | tail -5
print('installed')

In [ ]:
# --- Import smoke test. Five seconds here beats failing after the Drive mount, ---
# --- the 15 GB model download and the API key prompt.                          ---
import subprocess, sys
r = subprocess.run([sys.executable, '-c',
                    'from src.hipporag.HippoRAG import HippoRAG; print("imports ok")'],
                   cwd='/content/ysda-graph-rag/third_party/HippoRAG',
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-1500:])
if r.returncode:
    print('\n^ If this is a torchcodec/sentence-transformers error, the pin above did '
          'not take.\n  Runtime > Restart session, then re-run from the Drive cell.')
assert r.returncode == 0, 'HippoRAG does not import; fix this before indexing.'

In [ ]:
# --- Data. Ours is HippoRAG's own release, so the corpus matches theirs exactly. ---
!python scripts/download_data.py musique
!python scripts/check_hipporag_data.py musique

The gate above must report **100% shared** and *same corpus AND same question set*.
That is what makes recall comparable to the paper's R@5 = 74.7 rather than merely
similar-looking. If it does not, stop — every number after it would be measuring the
wrong thing.

In [ ]:
# --- DeepSeek: key, and a probe of the two things that break integrations ---
import getpass, os, json, requests
os.environ['OPENAI_API_KEY'] = getpass.getpass('DeepSeek API key: ')

LLM_BASE = 'https://api.deepseek.com/v1'
LLM_NAME = 'deepseek-v4-flash'

r = requests.post(f'{LLM_BASE}/chat/completions',
    headers={'Authorization': 'Bearer ' + os.environ['OPENAI_API_KEY']},
    json={'model': LLM_NAME, 'max_tokens': 64,
          'messages': [{'role': 'user',
                        'content': 'Reply only with JSON {"named_entities": ["a"]}'}]},
    timeout=60)
print('HTTP', r.status_code)
d = r.json()
assert 'choices' in d, d
m = d['choices'][0]['message']
print('model returned :', d.get('model'))
print('content        :', repr(m.get('content'))[:160])
print('reasoning field:', 'yes' if (m.get('reasoning_content') or m.get('reasoning')) else 'no')
print('usage          :', d.get('usage'))
# content must be a string. If it is None, the model reasons in a side channel and
# our salvage handles it -- but you want to know that now, not at 20k calls.
assert isinstance(m.get('content'), str) and m['content'].strip(), (
    'empty content: the model answers in a reasoning channel. The patch salvages it, '
    'but expect higher output-token cost.')
print('\nok')

## Index

The long step: ~23k extraction calls to DeepSeek, then the encoder passes over 11.7k
passages, ~100k entities and ~130k facts on the L4.

**Re-run this cell after any disconnect.** OpenIE results and the parquet stores are
on Drive, so it resumes. Do not add `--force_openie_from_scratch` unless you mean to
discard them.

`--reasoning_effort ""` leaves the request untouched: it exists for gpt-oss, and an
endpoint that does not know the parameter may reject the call.

In [ ]:
import os, subprocess, sys
R = '/content/ysda-graph-rag'
env = dict(os.environ, PYTHONPATH=R, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

cmd = [sys.executable, 'main.py',
       '--dataset', 'musique', '--data_dir', f'{R}/data',
       '--llm_base_url', LLM_BASE, '--llm_name', LLM_NAME,
       '--reasoning_effort', '',
       '--price_in_per_1k', '0.00014', '--price_out_per_1k', '0.00028',
       '--openie_max_workers', '32',
       '--embedding_name', 'nvidia/NV-Embed-v2',
       '--embedding_batch_size', '8',      # 24 GB card, not 40
       '--embedding_max_seq_len', '512',   # longest MuSiQue passage is ~400 tokens
       '--save_dir', SAVE_DIR,
       '--skip_qa', '--num_queries', '20',
       '--dump', f'{R}/out/hipporag_musique_smoke.json']

os.makedirs(f'{R}/out', exist_ok=True)
p = subprocess.Popen(cmd, cwd=f'{R}/third_party/HippoRAG', env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     bufsize=1)
for line in p.stdout:
    # httpx logs one line per call and the bars use \r; keep the output readable
    if 'HTTP Request' not in line:
        print(line.rstrip()[:200])
print('exit', p.wait())

In [ ]:
# --- Did it work? Three numbers decide. ---
import json, glob
d = json.load(open(f'{R}/out/hipporag_musique_smoke.json'))
unmapped = sum(1 for v in d.values() for i in v if i < 0)
print(f'{len(d)} questions, {unmapped} unmapped docs')
assert unmapped == 0, ('retrieved passages did not map back to our corpus — the '
                       'doc->pid identity broke, and recall would be meaningless')
print('stores on Drive:')
!du -sh {SAVE_DIR}/musique/* 2>/dev/null | head
print('\nunmapped=0 means the corpus identity held. Cost for the run is printed as '
      '"indexing tokens: ... cost: ..." in the cell above.')

## Next

The index is built and reusable — every arm below is retrieval only, minutes each.

```python
# full 1000 questions, the paper's configuration
--num_queries 0 --dump .../hipporag_musique_llm.json

# no LLM at all: the dense baseline (the paper's 69.7 R@5 reference)
--rag_type standard --dump .../hipporag_musique_dense.json

# what the LLM fact filter actually contributes
--rerank_mode norerank --dump .../hipporag_musique_norerank.json
```

Then score them together — never by globbing a directory, which silently intersects
every arm down to the smallest:

```bash
python scripts/score_qafd.py musique --baseline norerank \
  --runs out/hipporag_musique_llm.json out/hipporag_musique_dense.json \
         out/hipporag_musique_norerank.json
```

**The reproduction gate:** `llm` should beat `dense` by roughly the paper's +5.0 R@5.
`dense` is pure NV-Embed retrieval with no graph and no LLM, so it should land near
69.7 regardless of which model did the OpenIE — if it doesn't, the fault is in data or
scoring, not method.